# Large-Scale Portfolio Briefs Generation V2

This notebook runs Brief V2 batches for a portfolio of `RP_ENTITY_ID` values, monitors each batch to completion, retrieves generated bullets, previews the first 10 entities, and exports all bullet results to Excel.

## Step 1: Load Company Identifiers from CSV File

Read the portfolio CSV, locate the `RP_ENTITY_ID` column, remove empty or duplicate identifiers, and prepare the entity list used by the V2 batch APIs.

In [1]:
from __future__ import annotations

import json
import os
import time
import traceback
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from IPython.display import Markdown, display

JsonDict = dict[str, Any]

CSV_PATH = Path("static/data/US_100.csv")
df = pd.read_csv(CSV_PATH, dtype=str)

entity_id_column = next((c for c in df.columns if c.strip().upper() == "RP_ENTITY_ID"), None)
if entity_id_column is None:
    raise ValueError(f"RP_ENTITY_ID column not found in {CSV_PATH}")

ids: list[str] = (
    df[entity_id_column]
    .astype(str)
    .str.strip()
    .replace({"": None})
    .dropna()
    .drop_duplicates()
    .tolist()
)

print(f"Loaded {len(ids)} unique RP_ENTITY_ID values from {CSV_PATH}")

Loaded 107 unique RP_ENTITY_ID values from static\data\US_100.csv


## Step 2: Configure Brief V2 Batch Processing

Configure the V2 API endpoints, batch size, and report window. A 24-hour window is the recommended baseline: the V2 workflow uses a single-day `2026-02-01T00:00:00` to `2026-02-01T23:59:59` window. Wider windows degrade quality and cost and collapse the day-by-day novelty filtering; for longer historical ranges use `POST /api/v1/scan`, which splits the range into per-day windows automatically.

In [2]:
# Batch size follows the V1 notebook's production-oriented default.
BATCH_SIZE: int = 50
companies: list[str] = ids

API_BASE_URL: str = "http://localhost:8000"
RUN_PARALLEL_URL: str = f"{API_BASE_URL}/api/v1/batch/run-parallel"
BULLETS_URL: str = f"{API_BASE_URL}/api/v1/reports/bullets"


# A 24-hour window is the recommended baseline. Wider windows degrade quality and cost,
# and a single multi-day window yields one big run per entity instead of the intended
# day-by-day novelty filtering. For longer historical ranges use POST /api/v1/scan,
# which splits the range into per-day windows automatically.
FORCE_WINDOW_START: str = "2026-02-01T00:00:00"
FORCE_WINDOW_END: str = "2026-02-01T23:59:59"

POLL_INTERVAL_SECONDS: int = 30
BATCH_TIMEOUT_SECONDS: int = 3600

# Optional API key support. The service authenticates via the X-Api-Key header
# (only required when the service runs with PUBLIC_MODE on); local dev skips auth.
token = os.environ.get("API_TOKEN") or os.environ.get("TOKEN") or os.environ.get("API_KEY")
headers: dict[str, str] = {"X-Api-Key": token} if token else {}

print(f"Using {len(companies)} companies")
print(f"Batch size: {BATCH_SIZE}")
print(f"Report window: {FORCE_WINDOW_START} to {FORCE_WINDOW_END}")

Using 107 companies
Batch size: 50
Report window: 2026-02-01T00:00:00 to 2026-02-01T23:59:59


## Step 3: Set Output Folder and File Names

All V2 artifacts are written under `output/` with V2-specific file names so they do not overwrite V1 outputs.

In [3]:
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BRIEF_V2_BATCH_SUMMARY_FILE = OUTPUT_DIR / "brief_v2_batch_summaries.json"
BRIEF_V2_FIRST5_BULLETS_FILE = OUTPUT_DIR / "brief_v2_first5_bullets.json"
BRIEF_V2_ALL_BULLETS_FILE = OUTPUT_DIR / "brief_v2_all_bullets.json"
OUT_XLSX = OUTPUT_DIR / "portfolio_briefs_generation_v2.xlsx"

## Step 4: Define API and Formatting Helpers

These helpers keep the V2 API calls, polling logic, JSON persistence, and citation formatting reusable across the notebook.

In [4]:
def chunk_entities(entity_ids: list[str], batch_size: int) -> list[list[str]]:
    """Split entity identifiers into fixed-size batches."""
    return [entity_ids[start : start + batch_size] for start in range(0, len(entity_ids), batch_size)]


def request_json(method: str, url: str, *, timeout: int = 60, **kwargs: Any) -> JsonDict:
    """Call the local Brief V2 API and return parsed JSON with useful HTTP error context."""
    response = requests.request(method, url, headers=headers, timeout=timeout, **kwargs)
    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        message = f"{method} {url} failed with {response.status_code}: {response.text}"
        raise requests.HTTPError(message) from exc
    return response.json()


def status_url_for(batch_id: str) -> str:
    """Build the V2 batch status endpoint for a submitted batch."""
    return f"{API_BASE_URL}/api/v1/batch/parallel/{batch_id}/status"


def is_batch_complete(status_payload: JsonDict) -> bool:
    """Return True when all entities in a V2 batch have either succeeded or failed."""
    total = int(status_payload.get("total") or 0)
    succeeded = int(status_payload.get("succeeded") or 0)
    failed = int(status_payload.get("failed") or 0)
    running = int(status_payload.get("running") or 0)
    not_started = int(status_payload.get("not_started") or 0)

    if total > 0:
        return succeeded + failed >= total
    return running == 0 and not_started == 0 and succeeded + failed > 0


def status_summary(status_payload: JsonDict) -> str:
    """Format a compact status line for notebook progress output."""
    return (
        f"succeeded={status_payload.get('succeeded', 0)}, "
        f"failed={status_payload.get('failed', 0)}, "
        f"running={status_payload.get('running', 0)}, "
        f"not_started={status_payload.get('not_started', 0)}, "
        f"total={status_payload.get('total', 0)}"
    )


def save_json(path: Path, payload: Any) -> None:
    """Persist payload as indented UTF-8 JSON."""
    with path.open("w", encoding="utf-8") as file_handle:
        json.dump(payload, file_handle, ensure_ascii=False, indent=2)


def fetch_bullets_for_entities(entity_ids: list[str]) -> JsonDict:
    """Retrieve saved V2 bullets for a list of entities."""
    if not entity_ids:
        return {"results": [], "total_entities": 0, "total_bullets": 0}
    return request_json("POST", BULLETS_URL, json={"entity_ids": entity_ids}, timeout=180)

## Step 5: Run Brief V2 Batches Sequentially

Submit one `run-parallel` batch at a time, poll its status until every entity has either succeeded or failed, then move to the next batch. This matches the requested flow: once a batch is complete, start the next batch.

In [5]:
batch_summaries: list[JsonDict] = []
entity_batches = chunk_entities(companies, BATCH_SIZE)

print(f"Batch starting date and time: {datetime.now()}")

for batch_number, batch in enumerate(entity_batches, start=1):
    batch_start = (batch_number - 1) * BATCH_SIZE + 1
    batch_end = batch_start + len(batch) - 1
    payload_batch: JsonDict = {
        "entity_ids": batch,
        "force_window_start": FORCE_WINDOW_START,
        "force_window_end": FORCE_WINDOW_END,
    }

    summary: JsonDict = {
        "batch_number": batch_number,
        "batch_start": batch_start,
        "batch_end": batch_end,
        "entity_count": len(batch),
        "entity_ids": batch,
        "payload": payload_batch,
        "status": "submitted",
        "submitted_at_local": datetime.now().isoformat(),
    }

    try:
        print(f"Submitting batch {batch_start}-{batch_end} ({len(batch)} entities)...")
        submit_response = request_json("POST", RUN_PARALLEL_URL, json=payload_batch, timeout=180)
        summary["submission_response"] = submit_response
    except requests.RequestException as exc:
        print(f"Batch {batch_start}-{batch_end} submission failed: {exc}")
        traceback.print_exc()
        summary["status"] = "submit_failed"
        summary["error"] = str(exc)
        batch_summaries.append(summary)
        continue

    batch_id = str(submit_response.get("batch_id") or "").strip()
    if not batch_id:
        print(f"Batch {batch_start}-{batch_end} did not return a batch_id; skipping status polling.")
        summary["status"] = "missing_batch_id"
        batch_summaries.append(summary)
        continue

    summary["batch_id"] = batch_id
    summary["submitted_at"] = submit_response.get("submitted_at")
    summary["total"] = submit_response.get("total")

    final_status_response: JsonDict | None = None
    waited_seconds = 0

    while waited_seconds <= BATCH_TIMEOUT_SECONDS:
        try:
            status_response = request_json("GET", status_url_for(batch_id), timeout=60)
            final_status_response = status_response
            print(f"Batch {batch_start}-{batch_end} status: {status_summary(status_response)}")

            if is_batch_complete(status_response):
                failed = int(status_response.get("failed") or 0)
                summary["status"] = "completed_with_failures" if failed else "completed"
                break
        except requests.RequestException as exc:
            print(f"Status check error for batch {batch_id}: {exc}")
            summary.setdefault("status_errors", []).append(str(exc))

        time.sleep(POLL_INTERVAL_SECONDS)
        waited_seconds += POLL_INTERVAL_SECONDS

    if final_status_response is None:
        summary["status"] = "status_unavailable"
    elif not is_batch_complete(final_status_response):
        summary["status"] = "timeout"

    summary["waited_seconds"] = waited_seconds
    summary["completed_at_local"] = datetime.now().isoformat()
    summary["final_status_response"] = final_status_response
    batch_summaries.append(summary)

    print(f"Batch {batch_start}-{batch_end} finished with status: {summary['status']}")

save_json(BRIEF_V2_BATCH_SUMMARY_FILE, batch_summaries)

print(f"Saved {len(batch_summaries)} batch summaries to {BRIEF_V2_BATCH_SUMMARY_FILE}")
print(f"Batch completion date and time: {datetime.now()}")

Batch starting date and time: 2026-06-17 14:45:43.936852
Submitting batch 1-50 (50 entities)...
Batch 1-50 status: succeeded=0, failed=0, running=0, not_started=50, total=50
Batch 1-50 status: succeeded=17, failed=0, running=10, not_started=23, total=50
Batch 1-50 status: succeeded=19, failed=0, running=10, not_started=21, total=50
Batch 1-50 status: succeeded=35, failed=0, running=10, not_started=5, total=50
Batch 1-50 status: succeeded=48, failed=0, running=2, not_started=0, total=50
Batch 1-50 status: succeeded=50, failed=0, running=0, not_started=0, total=50
Batch 1-50 finished with status: completed
Submitting batch 51-100 (50 entities)...
Batch 51-100 status: succeeded=0, failed=0, running=0, not_started=50, total=50
Batch 51-100 status: succeeded=15, failed=0, running=10, not_started=25, total=50
Batch 51-100 status: succeeded=24, failed=0, running=10, not_started=16, total=50
Batch 51-100 status: succeeded=42, failed=0, running=8, not_started=0, total=50
Batch 51-100 status: su

## Step 6: Retrieve Bullets for the First 5 Entities

After batches complete, fetch bullets for the first 5 entities and save the raw response for inspection.

In [6]:
preview_entity_ids: list[str] = companies[:5]
first5_bullets_response = fetch_bullets_for_entities(preview_entity_ids)
save_json(BRIEF_V2_FIRST5_BULLETS_FILE, first5_bullets_response)

first5_results: list[JsonDict] = first5_bullets_response.get("results", []) or []

print(
    f"Retrieved bullets for {len(first5_results)} preview entities "
    f"with {first5_bullets_response.get('total_bullets', 0)} total bullets."
)
print(f"Saved preview bullets to {BRIEF_V2_FIRST5_BULLETS_FILE}")

Retrieved bullets for 5 preview entities with 2 total bullets.
Saved preview bullets to output\brief_v2_first5_bullets.json


## Step 7: Display First 5 Entity Bullet Results

Render the first 5 V2 bullet results in a notebook-friendly analyst view, including run metadata, bullet text, citations, and discard summaries.

In [7]:
def source_names_from_citation(citation: JsonDict) -> list[str]:
    """Extract the citation source name from V2 citation metadata."""
    source_name = str(citation.get("source_name") or "").strip()
    return [source_name] if source_name else []


def citations_table(citations: list[JsonDict]) -> pd.DataFrame:
    """Build a readable citation table for notebook preview output."""
    rows: list[JsonDict] = []

    for citation_number, citation in enumerate(citations, start=1):
        source_names = source_names_from_citation(citation)
        rows.append({
            "#": citation_number,
            "source": ", ".join(source_names),
            "headline": str(citation.get("headline") or "").strip(),
            "citation_id": str(citation.get("id") or "").strip(),
        })

    return pd.DataFrame(rows, columns=["#", "source", "headline", "citation_id"])


def count_items(value: Any) -> int:
    """Count list-like discard buckets from the V2 response."""
    return len(value) if isinstance(value, list) else 0


def render_v2_entity_result(entity_result: JsonDict, *, top_n: int | None = None) -> None:
    """Display one V2 entity result with runs, bullets, citations, and discard counts."""
    entity_id = entity_result.get("entity_id", "N/A")
    entity_name = entity_result.get("entity_name") or "Unknown"
    found = entity_result.get("found")
    total_runs = entity_result.get("total_runs", 0)
    total_bullets = entity_result.get("total_bullets", 0)

    display(Markdown(
        f"## {entity_name}\n"
        f"**RP_ENTITY_ID:** `{entity_id}`  |  **Found:** {found}  |  "
        f"**Runs:** {total_runs}  |  **Bullets:** {total_bullets}"
    ))

    runs = entity_result.get("runs", []) or []
    if not runs:
        display(Markdown("_No runs found for this entity._"))
        return

    for run_index, run in enumerate(runs, start=1):
        bullets = run.get("bullets", []) or []
        display(Markdown(
            f"### Run {run_index}: `{run.get('run_id', 'N/A')}`\n"
            f"**Window:** {run.get('report_window_start', 'N/A')} to {run.get('report_window_end', 'N/A')}  |  "
            f"**Created:** {run.get('run_created_at', 'N/A')}  |  "
            f"**Saved:** {run.get('bullets_saved', len(bullets))}  |  "
            f"**Discarded:** {run.get('bullets_discarded', 0)}"
        ))

        discard_summary = (
            f"**Discarded by relevance:** {count_items(run.get('discarded_by_relevance'))}  |  "
            f"**Discarded by grounding:** {count_items(run.get('discarded_by_grounding'))}  |  "
            f"**Discarded by novelty:** {count_items(run.get('discarded_by_novelty'))}"
        )
        display(Markdown(discard_summary))

        if not bullets:
            display(Markdown("_No bullet points found for this run._"))
            continue

        limit = top_n if top_n is not None else len(bullets)
        table_rows: list[JsonDict] = []

        for bullet_number, bullet in enumerate(bullets[:limit], start=1):
            text = str(bullet.get("text") or "").strip()
            citations = bullet.get("citations", []) or []
            citation_df = citations_table(citations)
            decision_line = (
                f"embedding_decision={bullet.get('embedding_decision', '')}, "
                f"search_action={bullet.get('search_action', '')}, "
                f"is_fully_novel={bullet.get('is_fully_novel', '')}"
            )

            display(Markdown(f"{bullet_number}. {text}\n\n**Decisions:** `{decision_line}`"))
            display(Markdown("**Citations:**"))
            if citation_df.empty:
                display(Markdown("_No citations found for this bullet._"))
            else:
                display(citation_df)

            table_rows.append({
                "trace_id": bullet.get("trace_id", ""),
                "bullet": text,
                "citation_count": len(citations),
                "citation_ids": "; ".join(str(citation.get("id") or "") for citation in citations),
                "embedding_decision": bullet.get("embedding_decision", ""),
                "search_action": bullet.get("search_action", ""),
                "is_fully_novel": bullet.get("is_fully_novel", ""),
            })

        display(Markdown("**Raw table (for copy/export):**"))
        display(pd.DataFrame(table_rows))


for preview_result in first5_results:
    render_v2_entity_result(preview_result)

## Costco Wholesale Corp.
**RP_ENTITY_ID:** `B8EF97`  |  **Found:** True  |  **Runs:** 1  |  **Bullets:** 2

### Run 1: `48b7c875-9e85-40d8-bc1a-83dafb296f55`
**Window:** 2026-02-01T00:00:00 to 2026-02-01T23:59:59  |  **Created:** 2026-06-17T12:46:58.524045  |  **Saved:** 2  |  **Discarded:** 9

**Discarded by relevance:** 0  |  **Discarded by grounding:** 0  |  **Discarded by novelty:** 9

1. Costco Wholesale Corp. issued a recall for 'Mini Beignets filled with Caramel' after discovering they were inadvertently packaged with chocolate hazelnut beignets containing undeclared tree nuts, affecting purchases made from January 16-30 across 22 states.

**Decisions:** `embedding_decision=keep, search_action=keep, is_fully_novel=True`

**Citations:**

...

_No bullet points found for this run._

## U.S. Bancorp Co.
**RP_ENTITY_ID:** `6166D1`  |  **Found:** True  |  **Runs:** 1  |  **Bullets:** 0

### Run 1: `978b6ded-b466-4c41-bf67-2a15431bcfb2`
**Window:** 2026-02-01T00:00:00 to 2026-02-01T23:59:59  |  **Created:** 2026-06-17T12:46:13.111612  |  **Saved:** 0  |  **Discarded:** 2

**Discarded by relevance:** 1  |  **Discarded by grounding:** 0  |  **Discarded by novelty:** 1

_No bullet points found for this run._

## Step 8: Retrieve All Bullets and Export to Excel

Fetch saved bullets for all entities in the same `BATCH_SIZE` chunks, persist the raw combined V2 response, flatten runs/bullets/citations, and export the results to Excel.

In [8]:
def source_names_from_citation_for_export(citation: JsonDict) -> list[str]:
    """Extract the citation source name from V2 citation metadata."""
    source_name = str(citation.get("source_name") or "").strip()
    return [source_name] if source_name else []


def citation_fields(citations: list[JsonDict]) -> JsonDict:
    """Flatten citation objects into semicolon-delimited Excel fields."""
    return {
        "citation_ids": "; ".join(str(citation.get("id") or "") for citation in citations),
        "citation_headlines": "; ".join(str(citation.get("headline") or "") for citation in citations),
        "citation_sources": "; ".join(
            ", ".join(source_names_from_citation_for_export(citation)) for citation in citations
        ),
        "citation_texts": "\n---\n".join(str(citation.get("text") or "") for citation in citations),
    }


def entity_run_base_row(entity_result: JsonDict, run: JsonDict | None = None) -> JsonDict:
    """Build common export columns for an entity/run pair."""
    run_payload = run or {}
    return {
        "rp_entity_id": entity_result.get("entity_id", ""),
        "entity_name": entity_result.get("entity_name", ""),
        "found": entity_result.get("found", ""),
        "total_runs": entity_result.get("total_runs", 0),
        "total_bullets": entity_result.get("total_bullets", 0),
        "run_id": run_payload.get("run_id", ""),
        "report_window_start": run_payload.get("report_window_start", ""),
        "report_window_end": run_payload.get("report_window_end", ""),
        "run_created_at": run_payload.get("run_created_at", ""),
        "run_bullet_count": run_payload.get("bullet_count", 0),
        "bullets_saved": run_payload.get("bullets_saved", 0),
        "bullets_discarded": run_payload.get("bullets_discarded", 0),
        "discarded_by_relevance_count": count_items(run_payload.get("discarded_by_relevance")),
        "discarded_by_grounding_count": count_items(run_payload.get("discarded_by_grounding")),
        "discarded_by_novelty_count": count_items(run_payload.get("discarded_by_novelty")),
    }


def flatten_v2_results(results: list[JsonDict]) -> pd.DataFrame:
    """Flatten V2 bullet results into one Excel-friendly row per bullet."""
    rows: list[JsonDict] = []

    for entity_result in results:
        runs = entity_result.get("runs", []) or []
        if not runs:
            rows.append({
                **entity_run_base_row(entity_result),
                "bullet_number": "",
                "trace_id": "",
                "bullet_text": "",
                "embedding_decision": "",
                "search_action": "",
                "is_fully_novel": "",
                "citation_count": 0,
                **citation_fields([]),
            })
            continue

        for run in runs:
            bullets = run.get("bullets", []) or []
            if not bullets:
                rows.append({
                    **entity_run_base_row(entity_result, run),
                    "bullet_number": "",
                    "trace_id": "",
                    "bullet_text": "",
                    "embedding_decision": "",
                    "search_action": "",
                    "is_fully_novel": "",
                    "citation_count": 0,
                    **citation_fields([]),
                })
                continue

            for bullet_number, bullet in enumerate(bullets, start=1):
                citations = bullet.get("citations", []) or []
                rows.append({
                    **entity_run_base_row(entity_result, run),
                    "bullet_number": bullet_number,
                    "trace_id": bullet.get("trace_id", ""),
                    "bullet_text": str(bullet.get("text") or "").strip(),
                    "embedding_decision": bullet.get("embedding_decision", ""),
                    "search_action": bullet.get("search_action", ""),
                    "is_fully_novel": bullet.get("is_fully_novel", ""),
                    "citation_count": len(citations),
                    **citation_fields(citations),
                })

    return pd.DataFrame(rows)


all_bullets_results: list[JsonDict] = []
all_bullets_responses: list[JsonDict] = []

for batch_number, batch in enumerate(chunk_entities(companies, BATCH_SIZE), start=1):
    batch_start = (batch_number - 1) * BATCH_SIZE + 1
    batch_end = batch_start + len(batch) - 1
    print(f"Retrieving bullets for entities {batch_start}-{batch_end}...")

    try:
        bullets_response = fetch_bullets_for_entities(batch)
    except requests.RequestException as exc:
        print(f"Bullet retrieval failed for entities {batch_start}-{batch_end}: {exc}")
        traceback.print_exc()
        all_bullets_responses.append({
            "batch_number": batch_number,
            "batch_start": batch_start,
            "batch_end": batch_end,
            "entity_ids": batch,
            "status": "failed",
            "error": str(exc),
        })
        continue

    batch_results = bullets_response.get("results", []) or []
    all_bullets_results.extend(batch_results)
    all_bullets_responses.append({
        "batch_number": batch_number,
        "batch_start": batch_start,
        "batch_end": batch_end,
        "entity_ids": batch,
        "status": "completed",
        "total_entities": bullets_response.get("total_entities", len(batch_results)),
        "total_bullets": bullets_response.get("total_bullets", 0),
        "response": bullets_response,
    })
    print(f"Retrieved {len(batch_results)} entities and {bullets_response.get('total_bullets', 0)} bullets.")

all_bullets_payload: JsonDict = {
    "retrieved_at_local": datetime.now().isoformat(),
    "batch_size": BATCH_SIZE,
    "total_entities": len(all_bullets_results),
    "total_bullets": sum(int(result.get("total_bullets") or 0) for result in all_bullets_results),
    "results": all_bullets_results,
    "batch_responses": all_bullets_responses,
}

save_json(BRIEF_V2_ALL_BULLETS_FILE, all_bullets_payload)

df_out = flatten_v2_results(all_bullets_results)

with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as writer:
    df_out.to_excel(writer, index=False, sheet_name="Briefs V2")
    worksheet = writer.sheets["Briefs V2"]
    worksheet.freeze_panes(1, 0)
    worksheet.autofilter(0, 0, max(len(df_out), 1), max(len(df_out.columns) - 1, 0))

    for column_index, column_name in enumerate(df_out.columns):
        max_width = max(len(str(column_name)), 12)
        if not df_out.empty:
            sample_width = df_out[column_name].astype(str).str.slice(0, 80).map(len).max()
            max_width = max(max_width, int(sample_width))
        worksheet.set_column(column_index, column_index, min(max_width + 2, 80))

print(f"Saved raw V2 bullet payload to {BRIEF_V2_ALL_BULLETS_FILE}")
print(f"Written {len(df_out)} rows to {OUT_XLSX}")
display(df_out.head(20))

Retrieving bullets for entities 1-50...
Retrieved 50 entities and 23 bullets.
Retrieving bullets for entities 51-100...
Retrieved 50 entities and 29 bullets.
Retrieving bullets for entities 101-107...
Retrieved 7 entities and 1 bullets.
Saved raw V2 bullet payload to output\brief_v2_all_bullets.json
Written 135 rows to output\portfolio_briefs_generation_v2.xlsx


,rp_entity_id,entity_name,found,total_runs,total_bullets,run_id,report_window_start,report_window_end,run_created_at,run_bullet_count,...,trace_id,bullet_text,embedding_decision,search_action,is_fully_novel,citation_count,citation_ids,citation_headlines,citation_sources,citation_texts
0,B8EF97,Costco Wholesale Corp.,True,1,2,48b7c875-9e85-40d8-bc1a-83dafb296f55,2026-02-01T00:00:00,2026-02-01T23:59:59,2026-06-17T12:46:58.524045,2,...,569bd802-8c41-4302-84ea-dcf45f186718,Costco Wholesale Corp. issued a recall for 'Mi...,keep,keep,True,2,CQS:4D8FA8989670BFBCDF7DFA75DAA7FE82-1; CQS:A5...,Costco issues recall notice for bakery item du...,AOL.com; FOX Business,Costco issued a recall notice for mislabeled b...
1,B8EF97,Costco Wholesale Corp.,True,1,2,48b7c875-9e85-40d8-bc1a-83dafb296f55,2026-02-01T00:00:00,2026-02-01T23:59:59,2026-06-17T12:46:58.524045,2,...,0660c5b2-a68d-4eed-9193-53e852200740,"Costco Wholesale Corp., which offered grocery ...",keep,rewrite,False,2,CQS:C5259DB44929637FC433F68138E095A9-2; CQS:9E...,Costco Stock Is Soaring. Is It Too Late to Buy...,Nasdaq; Yahoo! Finance,In the 2026 fiscal first quarter (ended Nov. 2...
2,BB07E4,Amphenol Corp.,True,1,0,d8d1bc3a-6dc6-4931-a69d-668854d2a27e,2026-02-01T00:00:00,2026-02-01T23:59:59,2026-06-17T12:45:46.131603,0,...,,,,,,0,,,,
3,3461CF,Moody's Corp.,True,1,0,b06751d2-89c8-48bc-a041-159767294d8b,2026-02-01T00:00:00,2026-02-01T23:59:59,2026-06-17T12:46:28.099123,0,...,,,,,,0,,,,
4,3DC887,Arista Networks Inc.,True,1,0,7ecf68fe-4617-4d35-9e6c-f514375c8a88,2026-02-01T00:00:00,2026-02-01T23:59:59,2026-06-17T12:45:52.626667,0,...,,,,,,0,,,,
5,6166D1,U.S. Bancorp Co.,True,1,0,978b6ded-b466-4c41-bf67-2a15431bcfb2,2026-02-01T00:00:00,2026-02-01T23:59:59,2026-06-17T12:46:13.111612,0,...,,,,,,0,,,,
6,D8F347,Union Pacific Corp.,True,1,0,225a2b6d-0901-419c-8bdd-c7de5013fdd2,2026-02-01T00:00:00,2026-02-01T23:59:59,2026-06-17T12:45:47.623596,0,...,,,,,,0,,,,
7,D3D781,Snowflake Inc.,True,1,3,931d4dae-dfc0-4192-ba59-ad4ad9f288fd,2026-02-01T00:00:00,2026-02-01T23:59:59,2026-06-17T12:46:48.057344,3,...,ad199e30-77b3-4719-82b6-d820041f962e,Snowflake Inc. remains loss making with a net ...,keep,keep,True,2,CQS:FA2831AF9FE5044BD952697145D51062-3; CQS:FA...,Snowflake Energy Push Tests Growth Outlook Aft...,Yahoo! Finance; Yahoo! Finance,&#9878; Simply Wall St Valuation: Shares are d...
8,D3D781,Snowflake Inc.,True,1,3,931d4dae-dfc0-4192-ba59-ad4ad9f288fd,2026-02-01T00:00:00,2026-02-01T23:59:59,2026-06-17T12:46:48.057344,3,...,97f1d575-0a49-4a26-b3f8-fdea77eaa3ea,Snowflake Inc. has seen a recent 11.1% decline...,keep,keep,True,2,CQS:FA2831AF9FE5044BD952697145D51062-3; CQS:FA...,Snowflake Energy Push Tests Growth Outlook Aft...,Yahoo! Finance; Yahoo! Finance,&#9878; Simply Wall St Valuation: Shares are d...
9,D3D781,Snowflake Inc.,True,1,3,931d4dae-dfc0-4192-ba59-ad4ad9f288fd,2026-02-01T00:00:00,2026-02-01T23:59:59,2026-06-17T12:46:48.057344,3,...,a9291de5-962d-4e07-9580-2da68284afd7,DA Davidson maintains a positive outlook on Sn...,keep,keep,True,1,CQS:9DC16625B2B7A5288248165D54D358E1-1,DA Davidson Reiterates Buy on Snowflake (SNOW)...,Yahoo! Finance,Snowflake Inc. (NYSE:SNOW) is one of the 10 AI...
